# 11. Real-Time Speech-to-Text — Continuous Recognition from Microphone (Speech SDK)

This notebook uses the classic Azure AI Speech SDK (`azure.cognitiveservices.speech`) to perform **continuous,
real-time speech recognition from the local microphone**, printing each recognized phrase as it's finalized.
This is architecturally very different from notebook 10's batch file transcription: audio streams in live, and
recognition results arrive asynchronously via **event callbacks** rather than as one returned object.

**⚠️ Cannot run as a notebook cell without a live microphone.** This script blocks in a `while not done:` loop
waiting for continuous microphone input and a stop signal (silence timeout, cancellation, or manual
interruption) — there's no defined end state without real audio input or you stopping it yourself. If you
don't have a local microphone available (e.g. running on a headless machine or remote kernel), **run the
original `11_real_time_speech.py` as a standalone script from a terminal on a machine with a microphone**
instead of executing the cells below inside this notebook process.

**Difficulty:** Intermediate


## Prerequisites

**Python packages:**
- `azure-cognitiveservices-speech` — **not yet in the repo's root `requirements.txt`**, install with:
  ```
  pip install azure-cognitiveservices-speech
  ```
- `python-dotenv` (already in `requirements.txt`)

**Azure resources required:**
- An Azure AI Speech resource with a key and endpoint

**Hardware required:**
- A local microphone / audio input device, and permission for Python to access it (OS-level microphone
  permission on macOS)

**Environment variables** (add to root `.env`):
```
AZURE_SPEECH_ENDPOINT=https://<your-speech-resource>.cognitiveservices.azure.com/
AZURE_SPEECH_KEY=<your-speech-resource-key>
```


## What You'll Learn

- The difference between **continuous** recognition (this notebook) and **single-shot** recognition
  (notebook 13's `recognize_once_async`)
- How Speech SDK recognition is event-driven: `.recognized`, `.session_stopped`, `.canceled` event handlers
  connected via `.connect(callback)`
- How to structure a polling loop that waits for an async, event-driven process to signal completion
- Why this pattern doesn't translate directly into a single notebook cell execution model


### Step 1 — Configure the Speech SDK and the microphone audio source

`SpeechConfig(subscription=..., endpoint=...)` configures auth and target resource for all subsequent Speech
SDK calls. `AudioConfig(use_default_microphone=True)` tells the SDK to capture from the OS's default audio
input device rather than a file. `SpeechRecognizer` ties the two together into a single recognizer object.

💡 **Exam tip:** `SpeechConfig` can be constructed either from `subscription` + `region` (a short Azure region
string like `"eastus2"` — see notebook 13) or `subscription` + `endpoint` (a full resource URL, as here) — know
both construction patterns; they're interchangeable ways of pointing the SDK at the same underlying resource.

🔄 **Alternatives:** Use `AudioConfig(filename="some.wav")` instead of `use_default_microphone=True` to run
this exact same continuous-recognition code against a file instead of a live mic — useful for reproducible
testing without needing to speak into a microphone every run.


In [ ]:
import os
import time
from dotenv import load_dotenv
import azure.cognitiveservices.speech as speechsdk

load_dotenv()

endpoint = os.environ["AZURE_SPEECH_ENDPOINT"]
api_key = os.environ["AZURE_SPEECH_KEY"]

speech_config = speechsdk.SpeechConfig(
    subscription=api_key,
    endpoint=endpoint
)

audio_config = speechsdk.audio.AudioConfig(use_default_microphone=True)
speech_recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)

### Step 2 — Wire up event handlers for continuous recognition

Continuous recognition is **event-driven**: rather than blocking until one result comes back (as single-shot
recognition does — notebook 13), you register callback functions and the SDK invokes them on a background
thread as events occur:
- `.recognized` fires each time a phrase is finalized (fully recognized) — here it just prints `evt.result.text`
- `.session_stopped` fires when the session ends normally
- `.canceled` fires if recognition is cancelled/errors out

Both `session_stopped` and `canceled` are wired to the same `stop_cb`, which calls
`speech_recognizer.stop_continuous_recognition()` and flips the `done` flag — that flag is what the polling
loop in Step 3 watches for.

💡 **Exam tip:** Continuous recognition also has a `.recognizing` event (not used here) that fires repeatedly
with *interim*, not-yet-finalized results as you speak — useful for live-captioning UIs that want to show text
updating in real time before the phrase is finalized. This script only handles `.recognized` (final results).

🔄 **Alternatives:** For a short, single utterance where you don't need continuous listening, use
`recognize_once_async()` instead (see notebook 13) — no event handlers needed, just one blocking call.


In [ ]:
done = False

def stop_cb(evt):
    speech_recognizer.stop_continuous_recognition()
    global done
    done = True

speech_recognizer.recognized.connect(lambda evt: print(evt.result.text))
speech_recognizer.session_stopped.connect(stop_cb)
speech_recognizer.canceled.connect(stop_cb)

### Step 3 — Start continuous recognition and wait for it to finish

`start_continuous_recognition()` kicks off background audio capture and recognition — it returns immediately,
which is why the script then polls `done` in a `time.sleep(0.5)` loop rather than blocking synchronously.
`done` only flips to `True` once `session_stopped` or `canceled` fires (e.g. after a silence timeout, or if you
interrupt the process).

**This is the cell that requires a live microphone and will hang indefinitely in a notebook kernel without
one, or without a way to trigger a stop event.** If you don't have a microphone attached, skip running this
cell and instead run `11_real_time_speech.py` directly from a terminal:
```
python 11_real_time_speech.py
```
where you can naturally end it by stopping speaking (session/silence timeout) or with Ctrl+C.

💡 **Exam tip:** Recognize this "start async operation, poll a flag set by an event callback" pattern as one
common way to bridge the Speech SDK's callback-based API into synchronous script flow — an alternative is to
use `threading.Event().wait()` instead of a manual sleep loop, but the sleep-loop pattern shown here is exactly
what's in the Microsoft Speech SDK quickstart samples.

🔄 **Alternatives:** Wrap this in an `asyncio`-friendly pattern using `threading.Event` for cleaner
signaling than a manual `time.sleep` poll loop, if integrating into a larger async application.


In [ ]:
# Requires a live microphone — will block until a stop/cancel event fires.
# If no microphone is available, run 11_real_time_speech.py as a standalone script instead.
speech_recognizer.start_continuous_recognition()
while not done:
    time.sleep(0.5)

## Summary

This notebook set up continuous, real-time speech recognition from the local microphone using the Speech SDK's
event-driven API: `.recognized` printed each finalized phrase, while `.session_stopped`/`.canceled` signaled
the main thread to stop. Unlike notebook 10's batch file transcription, this pattern is built for live,
open-ended audio streams — at the cost of needing an event-driven polling structure instead of a single
blocking call.


## Try It Yourself

1. **Easy:** Add a handler for the `.recognizing` event (interim results) that prints partial text as you
   speak, prefixed with `"...".`
2. **Intermediate:** Swap `AudioConfig(use_default_microphone=True)` for `AudioConfig(filename="conversation.wav")`
   so you can test the continuous-recognition event flow without a live microphone.
3. **Advanced:** Add a manual stop mechanism (e.g. reading from `input()` in a separate thread, or a maximum
   run-time timer) that calls `speech_recognizer.stop_continuous_recognition()` after N seconds, so the script
   terminates deterministically instead of relying only on silence/cancellation events.
